# Measuring Foreign Exposure: Value Added

This notebook contains `Stata 17 MP 8-core` code that produces Tables 1 and 2 with Value Added HP detrended and First Differences. Imbs & Pauwels, *Measuring Foreign Exposure*, JIE, 2025. Package requirements for the code are:
* `estout`
* `outreg2`

These packages can be installed in Stata with the `ssc install` command. It also uses custom **.do** files: `loadMultilateralData.do`, `processValueAdded.do`, and `makeTablesValueAdded.do` available from the repo.


### Working directory name

In [1]:
clear 
set more off
global dir `c(pwd)'

### Run `loadMultilateralData.do` and `processValueAdded.do` 

`loadMultilateralData.do` loads the data `multilateral.txt` produced my MATLAB in `matlab/output`.

`processValueAdded.do` prepares the variables required for the analysis presented in this notebook. This includes removing the Rest of the World (ROW) from the data, taking logs, constructing the relevant measures including foreign exposure measures, namely HOT, TiVA, Exports, and "Phi". It also defines three aggregate sectors: Agriculture (AGR), Manufacturing (MFG), and Services (SER).

<span style="color:red;"> Warning: The script runs an HP filter on log Real Value Added (PPP) which may take a little over a minute. </span>


In [2]:
quietly{
    do "./scripts/loadMultilateralData.do"
    cd ../../stata 
    do "./scripts/processValueAdded.do"
}
describe



. describe

Contains data
 Observations:        36,120                  
    Variables:            21                  
--------------------------------------------------------------------------------
Variable      Storage   Display    Value
    name         type    format    label      Variable label
--------------------------------------------------------------------------------
country_i       str3    %9s                   Country
sector_r        str19   %19s                  Sector
sectorcode_r    str3    %9s                   WIOT Sector Code
nacecode_r      str7    %9s                   
year            int     %ty                   Time
cross           float   %9.0g                 group(country_i sectorcode_r)
ctry_i          long    %8.0g      ctry_i     Country
sect_r          long    %19.0g     sect_r     Sector
sect_year       float   %9.0g                 group(code_r year)
ctry_year       float   %9.0g                 group(ctry_i year)
lrva_ppp        float   %9.0g    

**Notes**: 
* xgo = X/Gross Output, i.e., first order component of HOT.
* phi is divided by 100,000,000 for the legibility of the coefficients and standard errors in the regressions

## Regressions

The regression output presented below estimate:
$$
\ln \left({\textrm{VA}_{i,t}^{r}} \right)= \beta _{0} + \beta _{1} X_{i,t}^{r} +\varepsilon_{i,t}^{r}.
$$
where $X_{i,t}^{r}$ is sequentially $\textrm{HOT}_{i,t}^{r}$, $\textrm{HOT1}_{i,t}^{r}$, $ \textrm{Exports}_{i,t}^{r}$, $ \phi_{i,t}^{r}$,  $\tau_{i,t}^{r}(\textrm{VA})$, and finally all variables are regressed jointly. There are two variants of $\ln \left({\textrm{VA}_{i,t}^{r}} \right)$:
- HP detrended
- First difference

The variable $ \textrm{Exports}_{i,t}^{r}$ is either measured as Exports/Gross Value Added.

## Produce LaTeX tables

The file `makeTablesValueAdded.do` uses `outreg2` to produce all the LaTeX tables required for Table 1 and 2.

In [3]:
quietly do "`dir'./scripts/makeTablesValueAdded.do"


 :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :
>   :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  :  : 


### Relevant R2

In [3]:
xi: reg dlrva_ppp hot if mfg_r == 1,  vce(cluster cross)
xi: reg dlrva_ppp xgo if mfg_r == 1,  vce(cluster cross)



Linear regression                               Number of obs     =     13,270
                                                F(1, 884)         =       7.02
                                                Prob > F          =     0.0082
                                                R-squared         =     0.0001
                                                Root MSE          =     .14041

                                (Std. err. adjusted for 885 clusters in cross)
------------------------------------------------------------------------------
             |               Robust
   dlrva_ppp | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
         hot |   .0061164   .0023091     2.65   0.008     .0015844    .0106484
       _cons |  -.0032729   .0012247    -2.67   0.008    -.0056766   -.0008692
------------------------------------------------------------------------------


Linear regr

In [4]:
xi: reg d.lrva_ppp d.hot if mfg_r == 1,  vce(cluster cross)
xi: reg d.lrva_ppp d.xgo if mfg_r == 1,  vce(cluster cross)



Linear regression                               Number of obs     =     12,383
                                                F(1, 884)         =      12.24
                                                Prob > F          =     0.0005
                                                R-squared         =     0.0043
                                                Root MSE          =     .18001

                                (Std. err. adjusted for 885 clusters in cross)
------------------------------------------------------------------------------
             |               Robust
  D.lrva_ppp | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
         hot |
         D1. |    .239406   .0684209     3.50   0.000     .1051196    .3736924
             |
       _cons |   .0005796   .0018128     0.32   0.749    -.0029783    .0041376
--------------------------------------------------------------